## **Nugen Intelligence**
<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="200"/>

Domain-aligned foundational models at industry leading speeds and zero-data retention! To learn more, visit [Nugen](https://docs.nugen.in/introduction)

### **Vision Alignmeent with the Nugen API**
 
This cookbook demonstrates how to create a Vision Alignment Project using the Nugen API. You'll learn how to upload an image dataset, automatically generate benchmark questions, train an aligned vision model, monitor training progress, and finally perform inference using the aligned model.

The notebook explains each step in a simple, sequential manner so that you can easily reproduce the complete workflow.

### **Dataset Preparation**

Before creating a Vision Alignment project, your image dataset must be preprocessed.

- Convert every image to a **Base64-encoded string**. 
  Eample-{"image": "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAKAAAAAoCAIAAAD2TmbPAAAL}
- Create a **`.jsonl`** file containing one JSON object per line.
- Each JSON object should include the Base64-encoded image and the corresponding annotation or metadata required for training.
- Save the file with the `.jsonl` extension.
- Use this `.jsonl` file when uploading the dataset for Vision Alignment.

> **Note:** Vision Alignment accepts image datasets in Base64 format stored in a `.jsonl` file. Raw image files (such as `.jpg` or `.png`) must be converted before starting the alignment process.

### **Workflow**                                                                                  
The cookbook covers the following steps:

1 Upload a vision dataset (.jsonl).
2 Retrieve document details.
3 Generate benchmark questions from the uploaded dataset.
4 Create a Vision Alignment project.
5 Monitor alignment training status.
6 Run inference using the aligned vision model.

### **Dataset Split**

After uploading your dataset, 15% of the uploaded data will automatically be used to generate benchmark questions for evaluating the aligned model. The remaining data is used during the alignment process.

Uploaded Dataset
      - 85% → Alignment Training
      - 15% → Benchmark Generation

### **Step 1**

**Install the required Python packages**

In [1]:
!pip install --quiet requests pandas python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


**Import Required Libraries**

In [2]:
import os 
import requests
import pandas as pd
import json
import time
import base64
from dotenv import load_dotenv
load_dotenv()

True

**Set up the Nugen API Client**

To read more about Nugen API and access free API keys, you can visit [Nugen Dashboard](https://platform.nugen.in/)

**Configure the API Client**

Set your Nugen API key.

In [3]:
api_key = os.getenv("NUGEN_API_KEY")

In [4]:
headers = {"Authorization": f"Bearer {api_key}"}

Here, we define the API base URL and your API key. Replace <--nugen api key--> with your actual key to authenticate your requests to the Nugen API. The MODEL variable specifies the model we will use for generating the routines.

In [5]:
BASE_URL = "https://api.nugen.in"

### **Step 2**

**Upload the Vision Dataset**

Upload a JSONL dataset

In [6]:
url = f"{BASE_URL}/api/v3/documents"

In [ ]:
with open("vision_dataset.jsonl", "rb") as f:
    files = {
        "files": ("vision_dataset.jsonl", f, "application/json")
    }

    data = {
        "categories": "image"
    }

    document_response = requests.post(
        url,
        headers=headers,
        data=data,
        files=files
    )
print(document_response.text)


{"document_ids":["01KX2Q5NV39D2FV"]}


**Get status of Document uploaded**

Once the upload is complete, retrieve the document ID

This endpoint returns metadata such as the document ID and processing status.

In [8]:
response_data = json.loads(document_response.text) 

id = response_data["document_ids"][0]

In [9]:
document_status_url = f"{BASE_URL}/api/v3/documents/{id}"

In [10]:
document_status_response = requests.get(document_status_url, headers=headers)

print(document_status_response.text)

{"status":"READY","document_id":"doc_01KX2Q5P19G7T4J"}


**Generate Benchmark Questions**

Generate evaluation questions from the uploaded dataset.

Note: Benchmark questions are generated using 15% of the uploaded dataset

In [11]:
response_data = json.loads(document_status_response.text) 

document_id = response_data["document_id"]

In [12]:
generate_benchmark_url = f"{BASE_URL}/api/v3/benchmark/create"

In [13]:
payload = {
    "documents": [document_id],
    "num_questions": 20
}

benchmark_response = requests.post(generate_benchmark_url, json=payload, headers=headers)

print(benchmark_response.text)

{"benchmark_id":"benchmark_01KX2Q6BCW9TBDN","status":"PROCESSING"}


**Get status Benchmark Generation**

Check whether benchmark generation has completed.

In [14]:
response_data = json.loads(benchmark_response.text)

benchmark_id = response_data["benchmark_id"]

In [15]:
benchmark_status_url = f"{BASE_URL}/api/v3/benchmark/status/{benchmark_id}"

In [17]:
response = requests.get(benchmark_status_url, headers=headers)

print(response.text)

{"benchmark_id":"benchmark_01KX2Q6BCW9TBDN","benchmark_name":"generated_benchmark_01KX2Q6BCW9TBDN","status":"READY","start_time":"2026-07-09T05:56:44.031988","end_time":"2026-07-09T05:56:44.164315"}


**Create a Vision Alignment Project**

**Example base model:**

qwen2-vl-2b-instruct

In [48]:
alignment_url = f"{BASE_URL}/api/v3/alignment-project/create"

In [ ]:
payload = {
    "name": "My Vision Alignment ",
    "base_model": "qwen2-vl-2b-instruct",
    "document_ids": [document_id],
    "workflow_id": "workflow-abc123",
    "benchmark_id": benchmark_id,
    "description": "This project aims to align the model for better vision alignment."
}

print(payload)
alignment_response = requests.post(alignment_url, json=payload, headers=headers)

print(alignment_response.text)

{'name': 'My Vision Alignment testing', 'base_model': 'qwen2-vl-2b-instruct', 'document_ids': ['doc_01KWGZ3G6N9BVVT'], 'workflow_id': 'workflow-abc123', 'benchmark_id': 'benchmark_01KX0K7YKFJW1KT', 'description': 'This project aims to align the model for better vision alignment.'}
{"alignment_id":"alignment_01KX0M3CWEN5107","status":"PROCESSING"}


**Check Alignment Status**

Track the alignment status.

In [ ]:
response_data = json.loads(alignment_response.text)

alignment_id = response_data["alignment_id"]

In [ ]:
alignment_status_url = f"{BASE_URL}/api/v3/alignment-project/status/{alignment_id}"

In [ ]:
while True:
    alignment_response = requests.get(alignment_status_url, headers=headers)
    alignment_response.raise_for_status()
    print(alignment_response.text)
    data = alignment_response.json()
    status = data["status"]

    print(f"Current status of Alignment: {status}")

    if status == "READY":
        print("Alignment completed.")
        break

    if status == "FAILED":
        raise Exception("Alignment failed.")
    time.sleep(10)

**Deploy Aligned Model**

Once alignment completes, you'll be able to deploy the model.


In [18]:
response_data = json.loads(alignment_response.text)

model_id = response_data["data"]["model_id"]

In [ ]:
deploy_url = f"{BASE_URL}/api/v3/models/deploy-model/{model_id}"

In [67]:
deploy_response = requests.post(deploy_url, headers=headers)

print(deploy_response.text)

{"model_id":"my-vision-alignmentalignment-01kx0619c87yr5s"}


**Deploy Status**

Check the deployment status of an aligned model.

In [68]:
response_data = json.loads(deploy_response.text)

model_id = response_data["model_id"]

In [69]:
deploy_status_url = f"https://api.nugen.in/api/v3/models/deploy-model/{model_id}/status"

In [70]:
response = requests.get(deploy_status_url, headers=headers)

print(response.text)

{"model_id":"my-vision-alignmentalignment-01kx0619c87yr5s","status":"COMPLETED","result":{"deployment_status":"DEPLOYED"},"start_time":"2026-07-08T11:16:18.493963","end_time":"2026-07-08T11:16:18.498291"}


**Run Inference**

Once alignment completes, you'll receive an Aligned Model ID.
Use this model for inference.

In [15]:
inference_url = f"{BASE_URL}/api/v3/inference/chat/completions"

In [ ]:
with open("vision_image.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

payload = {
    "model": model_id,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "tell me about this image"},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]
        }
    ],
    "max_tokens": 100,
    "temperature": 1,
    "stream": True,
}

In [ ]:
response = requests.post(inference_url, headers=headers, json=payload)

full_content = ""
usage = None

for line in response.iter_lines(decode_unicode=True):
    if not line:
        continue

    if not line.startswith("data: "):
        continue

    data = line[6:]

    if data == "[DONE]":
        continue

    try:
        chunk = json.loads(data)

        # Collect generated content
        choices = chunk.get("choices", [])

        if choices:
            delta = choices[0].get("delta", {})
            content = delta.get("content", "")

            if content:
                full_content += content

            # Collect final usage
            if chunk.get("usage"):
                usage = chunk["usage"]

    except json.JSONDecodeError:
        continue

print("content:", full_content)
print("usage:", usage)

**Explanation**

Vision Alignment enables you to create a Vision Alignment Model using your own image dataset, allowing the model to better understand and respond to domain-specific visual content. Instead of relying solely on the model's general knowledge, alignment adapts the model to your organization's data and use case.

The Vision Alignment workflow in this cookbook consists of the following steps:

1. Upload a vision dataset in JSONL format.
2. Generate benchmark questions using **15% of the uploaded dataset** to evaluate the aligned model.
3. Use the remaining **85% of the dataset** for alignment training.
4. Create an alignment project by selecting an alignment-ready Vision Language Model.
5. Monitor the training progress until the alignment is complete.
6. Perform inference using the newly aligned model.

By following this workflow, you can create a vision model that produces more accurate and context-aware responses for your specific application.

**Conclusion**

In this cookbook, you learned how to build a complete Vision Alignment pipeline using the Nugen API. Starting with a vision dataset, you uploaded the data, generated benchmark questions-answers, created an alignment project, monitored the alignment process, and finally used the aligned model for inference.

This workflow provides a simple and effective way to adapt a Vision Language Model to domain-specific image understanding tasks.